# CP468 Assignment 2 – Q-Learning Parking Problem
Goal: The agent starts at S and must choose a route to a parking lot. It should learn to minimize travel time, parking cost, and walking distance, while avoiding unavailable or invalid parking options.

## Environment Graph Setup

In [ ]:
import random
import matplotlib.pyplot as plt
from math import inf
import heapq
# Graph: node -> {action: (next_state, travel_time)}
graph = {
    "S": {"A": ("A", 4), "B": ("B", 2)},
    "A": {"C": ("C", 3), "P1": ("P1", 6)},
    "B": {"A": ("A", 1), "P2": ("P2", 5)},
    "C": {"P1": ("P1", 2), "P2": ("P2", 2)},
    "P1": {"park": ("TERMINAL", 0)},
    "P2": {"park": ("TERMINAL", 0)},
    "TERMINAL": {}
}



Task 1: MDP Formulation

States: Current node the agent occupies — {S, A, B, C, P1, P2, TERMINAL} Actions: From each node, move to an adjacent node via a directed edge. At parking nodes P1/P2, the agent may take the park action.

Reward:

Move: R = -travel_time
Park: R = -(cost + beta × walk_dist) if lot is feasible- Park: R = -1000 if lot is unavailable or walk_dist > max_walk
Objective: Maximise cumulative reward == minimise total driving + parking cost, subject to the walking distance constraint.

## Markov Decision Process

In [ ]:
states = list(graph.keys())
actions = {state: list(graph[state].keys()) for state in graph}

print("States:")
print(states)

print("\nActions:")
for state in actions:
    print(f"{state}: {actions[state]}")

States:
['S', 'A', 'B', 'C', 'P1', 'P2', 'TERMINAL']

Actions:
S: ['A', 'B']
A: ['C', 'P1']
B: ['A', 'P2']
C: ['P1', 'P2']
P1: ['park']
P2: ['park']
TERMINAL: []


## Parameters

In [ ]:
alpha = 0.1      # learning rate
gamma = 0.9      # discount factor
epsilon = 0.2    # exploration rate
episodes = 5000
beta = 0.01      # weight for walking distance
start_state = "S"
terminal_state = "TERMINAL"

## Q-table Initialization

In [ ]:
def initialize_q(graph):
    Q = {}
    for state in graph:
        Q[state] = {}
        for action in graph[state]:
            Q[state][action] = 0.0
    return Q

## Reward Function

In [ ]:
def get_reward(state, action, next_state, graph, parking_lots, beta=0.01):
    # Parking action
    if action == "park":
        lot_info = parking_lots.get(state)

        if lot_info is None:
          return -1000 #invalid parking lot

        if (not lot_info["available"]) or (lot_info["walk_dist"] > lot_info["max_walk"]):
            return -1000

        return -(lot_info["cost"] + beta * lot_info["walk_dist"])

    # Movement action
    if state in graph and action in graph[state]:
      _, travel_time = graph[state][action]
      return -travel_time

    return -1000 # Invalid action

## Action Selection

In [ ]:
def choose_action(state, Q, actions, epsilon):
    if random.random() < epsilon:
        return random.choice(actions[state])
    return max(Q[state], key=Q[state].get)

## Q-Learning Training

In [ ]:
def train_q_learning(graph, parking_lots, episodes, alpha, gamma, epsilon, beta=0.01,
                     start_state="S", terminal_state="TERMINAL"):

    actions = {state: list(graph[state].keys()) for state in graph}
    Q = initialize_q(graph)
    rewards = [] #Track convergence
    current_epsilon = epsilon

    for episode in range(episodes):
        state = start_state
        episode_reward = 0

        while state != terminal_state:
            action = choose_action(state, Q, actions, current_epsilon)
            next_state, _ = graph[state][action]

            reward = get_reward(state, action, next_state, graph, parking_lots, beta)

            episode_reward += reward

            if next_state == terminal_state:
                max_future_q = 0
            else:
                max_future_q = max(Q[next_state].values()) if Q[next_state] else 0

            Q[state][action] = Q[state][action] + alpha * (
                reward + gamma * max_future_q - Q[state][action]
            )

            state = next_state

        rewards.append(episode_reward)

        #Decay epsilon after each episode
        current_epsilon = max(0.01, current_epsilon * 0.999)

        # Progress tracking
        if (episode + 1) % 1000 == 0:
            avg_reward = sum(rewards[-100:]) / 100
            print(f"Episode {episode + 1}/{episodes}, Epsilon: {current_epsilon:.4f}, Avg Reward (last 100): {avg_reward:.2f}")

    return Q, rewards

## Policy Selection

In [ ]:
def extract_policy(Q):
    policy = {}
    for state in Q:
        if Q[state]:
            policy[state] = max(Q[state], key=Q[state].get)
    return policy

# Results

In [ ]:
def print_q_table(Q):
    print("Learned Q-values: (best action per state):")
    for state in Q:
      if Q[state]:
        best_action = max(Q[state], key=Q[state].get)
        print(f"{state}: best = {best_action}, value = {Q[state][best_action]:.2f}")

def print_policy(policy):
    print("\nLearned policy:")
    for state in policy:
        print(f"{state} -> {policy[state]}")


## Simulate Route

In [ ]:
def simulate_policy(graph, policy, start_state="S", terminal_state="TERMINAL"):
    state = start_state
    path = [state]

    while state != terminal_state:
        action = policy[state]
        next_state, _ = graph[state][action]
        path.append(next_state)
        state = next_state

    return path

# Testing

### Test Function

In [ ]:
def run_test_case(test_name, graph, parking_lots, episodes, alpha, gamma, epsilon, beta=0.01):
    print("=" * 50)
    print(f"TEST CASE: {test_name}")
    print("=" * 50)

    Q,  rewards = train_q_learning(
        graph=graph,
        parking_lots=parking_lots,
        episodes=episodes,
        alpha=alpha,
        gamma=gamma,
        epsilon=epsilon,
        beta=beta
    )

    policy = extract_policy(Q)
    path = simulate_policy(graph, policy)

    print_q_table(Q)
    print_policy(policy)

    print("\nChosen path:")
    print(" -> ".join(path))

    print("\nChosen parking lot:", path[-2])

    print("\nParking lot info:")
    for lot in parking_lots:
        print(f"{lot}: {parking_lots[lot]}")

    # Calculate total cost
    total_cost = 0
    for i in range(len(path)-1):
        if path[i] in graph and path[i+1] in [node for node in graph[path[i]].keys()]:
            if path[i+1] != "TERMINAL":
                _, travel_time = graph[path[i]][path[i+1]]
                total_cost += travel_time
            else:
                lot = path[i]
                lot_info = parking_lots.get(lot)
                if lot_info:
                    total_cost += lot_info["cost"] + beta * lot_info["walk_dist"]

    print(f"\nTotal cost: {total_cost:.2f}")

    print("\n")
    return Q, policy, path, rewards

## Test 1: Baseline

In [ ]:
parking_lots_1 = {
    "P1": {"cost": 8, "walk_dist": 350, "available": True, "max_walk": 500},
    "P2": {"cost": 3, "walk_dist": 450, "available": True, "max_walk": 500}
}

Q1, policy1, path1, rewards1 = run_test_case(
    test_name="Baseline",
    graph=graph,
    parking_lots=parking_lots_1,
    episodes=episodes,
    alpha=alpha,
    gamma=gamma,
    epsilon=epsilon,
    beta=beta
)

TEST CASE: Baseline
Episode 1000/5000, Epsilon: 0.0735, Avg Reward (last 100): -15.71
Episode 2000/5000, Epsilon: 0.0270, Avg Reward (last 100): -15.52
Episode 3000/5000, Epsilon: 0.0100, Avg Reward (last 100): -15.54
Episode 4000/5000, Epsilon: 0.0100, Avg Reward (last 100): -15.60
Episode 5000/5000, Epsilon: 0.0100, Avg Reward (last 100): -15.50
Learned Q-values: (best action per state):
S: best = B, value = -11.71
A: best = C, value = -10.87
B: best = A, value = -10.79
C: best = P2, value = -8.75
P1: best = park, value = -11.50
P2: best = park, value = -7.50

Learned policy:
S -> B
A -> C
B -> A
C -> P2
P1 -> park
P2 -> park

Chosen path:
S -> B -> A -> C -> P2 -> TERMINAL

Chosen parking lot: P2

Parking lot info:
P1: {'cost': 8, 'walk_dist': 350, 'available': True, 'max_walk': 500}
P2: {'cost': 3, 'walk_dist': 450, 'available': True, 'max_walk': 500}

Total cost: 8.00




## Test 2: P2 Too Far

In [ ]:
parking_lots_2 = {
    "P1": {"cost": 8, "walk_dist": 350, "available": True, "max_walk": 500},
    "P2": {"cost": 3, "walk_dist": 600, "available": True, "max_walk": 500}
}

Q2, policy2, path2, rewards2 = run_test_case(
    test_name="P2 Too Far",
    graph=graph,
    parking_lots=parking_lots_2,
    episodes=episodes,
    alpha=alpha,
    gamma=gamma,
    epsilon=epsilon,
    beta=beta
)

TEST CASE: P2 Too Far
Episode 1000/5000, Epsilon: 0.0735, Avg Reward (last 100): -98.65
Episode 2000/5000, Epsilon: 0.0270, Avg Reward (last 100): -49.15
Episode 3000/5000, Epsilon: 0.0100, Avg Reward (last 100): -19.50
Episode 4000/5000, Epsilon: 0.0100, Avg Reward (last 100): -19.52
Episode 5000/5000, Epsilon: 0.0100, Avg Reward (last 100): -19.51
Learned Q-values: (best action per state):
S: best = B, value = -14.33
A: best = C, value = -14.11
B: best = A, value = -13.70
C: best = P1, value = -12.35
P1: best = park, value = -11.50
P2: best = park, value = -1000.00

Learned policy:
S -> B
A -> C
B -> A
C -> P1
P1 -> park
P2 -> park

Chosen path:
S -> B -> A -> C -> P1 -> TERMINAL

Chosen parking lot: P1

Parking lot info:
P1: {'cost': 8, 'walk_dist': 350, 'available': True, 'max_walk': 500}
P2: {'cost': 3, 'walk_dist': 600, 'available': True, 'max_walk': 500}

Total cost: 8.00




## Test 3: P1 Unavailable

In [ ]:
parking_lots_3 = {
    "P1": {"cost": 8, "walk_dist": 350, "available": False, "max_walk": 500},
    "P2": {"cost": 3, "walk_dist": 450, "available": True, "max_walk": 500}
}

Q3, policy3, path3, rewards3 = run_test_case(
    test_name="P1 Unavailable",
    graph=graph,
    parking_lots=parking_lots_3,
    episodes=episodes,
    alpha=alpha,
    gamma=gamma,
    epsilon=epsilon,
    beta=beta
)

TEST CASE: P1 Unavailable
Episode 1000/5000, Epsilon: 0.0735, Avg Reward (last 100): -75.07
Episode 2000/5000, Epsilon: 0.0270, Avg Reward (last 100): -55.26
Episode 3000/5000, Epsilon: 0.0100, Avg Reward (last 100): -55.22
Episode 4000/5000, Epsilon: 0.0100, Avg Reward (last 100): -25.41
Episode 5000/5000, Epsilon: 0.0100, Avg Reward (last 100): -25.45
Learned Q-values: (best action per state):
S: best = B, value = -11.71
A: best = C, value = -10.87
B: best = A, value = -10.79
C: best = P2, value = -8.75
P1: best = park, value = -1000.00
P2: best = park, value = -7.50

Learned policy:
S -> B
A -> C
B -> A
C -> P2
P1 -> park
P2 -> park

Chosen path:
S -> B -> A -> C -> P2 -> TERMINAL

Chosen parking lot: P2

Parking lot info:
P1: {'cost': 8, 'walk_dist': 350, 'available': False, 'max_walk': 500}
P2: {'cost': 3, 'walk_dist': 450, 'available': True, 'max_walk': 500}

Total cost: 8.00


